In [ ]:
import re
import json
import pandas as pd
import numpy as np
from google.colab import files
from sklearn.metrics import f1_score as sklearn_f1_score

# **Chuyển định dạng đầu vào**

In [ ]:
aspects = [
    'hotel#general', 'hotel#prices', 'hotel#design&features', 'hotel#cleanliness', 'hotel#comfort', 'hotel#quality', 'hotel#miscellaneous',
    'rooms#general', 'rooms#prices', 'rooms#design&features', 'rooms#cleanliness', 'rooms#comfort', 'rooms#quality', 'rooms#miscellaneous',
    'room_amenities#general', 'room_amenities#prices', 'room_amenities#design&features', 'room_amenities#cleanliness', 'room_amenities#comfort', 'room_amenities#quality', 'room_amenities#miscellaneous',
    'facilities#general', 'facilities#prices', 'facilities#design&features', 'facilities#cleanliness', 'facilities#comfort', 'facilities#quality', 'facilities#miscellaneous',
    'service#general',
    'location#general',
    'food&drinks#prices', 'food&drinks#quality', 'food&drinks#style&options', 'food&drinks#miscellaneous' ]

all_keys = aspects

In [ ]:
# Chuyển định dạng reviews từ file .txt -> Dataframe
def text2df(text):
    blocks = re.split(r'(?m)^#\d+\s*', text.strip())
    blocks = [b.strip() for b in blocks if b.strip()]

    data, found_keys = [], set()
    for b in blocks:
        parts = b.rsplit('\n', 1)
        review, ann_line = parts if len(parts) == 2 else (b, "")
        anns = re.findall(r'\{([^,{}]+),\s*([^{}]+)\}', ann_line)
        anns = [(a.strip().lower(), s.strip().lower()) for a, s in anns]
        found_keys.update(a for a, _ in anns)
        data.append((review.strip(), anns))

    rows = []
    for review, anns in data:
        row = {"review": review}
        for k in all_keys:
            row[k] = np.nan
        for a, s in anns:
            if a in row:
                row[a] = s
        rows.append(row)

    return pd.DataFrame(rows), True

In [ ]:
# Chuyển định dạng reviews từ file .json -> Dataframe
def json2df(data):
    rows = []
    for d in data:
        row = {"review": (d.get("review") or "").strip()}
        # mặc định NaN cho tất cả aspect
        for asp in all_keys:
            row[asp] = np.nan
        # điền sentiment theo annotations
        for ann in d.get("annotations", []):
            asp, sent = ann.get("aspect"), ann.get("sentiment")
            if asp in row:
                row[asp] = sent
        rows.append(row)
    df = pd.DataFrame(rows)
    return df, True

# **Độ đo IAA F1-score**

In [ ]:
# Chuyển ma trận (N, K) thành ma trận nhị phân (N, 3*K) để tính F1-score
def multioutput_to_multilabel(y_sentiment_indices):
    if isinstance(y_sentiment_indices, pd.DataFrame):
        y_sentiment_indices = y_sentiment_indices.values

    nrow = y_sentiment_indices.shape[0] # Số lượng mẫu.
    ncol = y_sentiment_indices.shape[1] # Số lượng aspect.

    # Khởi tạo mảng Multi-label (Boolean) với kích thước: Hàng x (3 * Cột).
    multilabel = np.zeros((nrow, 3 * ncol), dtype=bool)
    for i in range(nrow):
        for j in range(ncol):
            sentiment_idx = y_sentiment_indices[i, j]
            if sentiment_idx != 0:
                pos = j * 3 + (sentiment_idx - 1)
                multilabel[i, pos] = True
    return multilabel

# Tính F1-score dựa trên ma trận nhị phân
def custom_f1_score(y_true, y_pred, average='micro', **kwargs):
    y_true_ml = multioutput_to_multilabel(y_true)
    y_pred_ml = multioutput_to_multilabel(y_pred)
    return sklearn_f1_score(y_true_ml, y_pred_ml, average=average, **kwargs)

In [ ]:
def calc_IAA(file1, file2, file3, file4 = None, all_keys=all_keys, text=False):
    file_list = [file1, file2, file3]
    if file4 is not None:
        file_list.append(file4)

    dataframes = []
    labeled_flags = []

    for i, file_path in enumerate(file_list):
        try:
            if i < 3:
                df, labeled = json2df(file_path) if text==False else text2df(file_path)
            else:
                df, labeled = text2df(file_path)
            dataframes.append(df)
            labeled_flags.append(labeled)
        except Exception as e:
            print(f"Lỗi khi phân tích cú pháp file {i+1} ({file_path}): {e}")
            return

    df1, df2, df3 = dataframes[0], dataframes[1], dataframes[2]
    labeled1, labeled2, labeled3 = labeled_flags[0], labeled_flags[1], labeled_flags[2]

    if not all(labeled_flags):
        print("All files must be labeled format.")
        return

    review_cols = [df['review'].astype(str) for df in dataframes]
    lengths = [len(df) for df in dataframes]

    if not all(l == lengths[0] for l in lengths):
        print("Review texts or counts differ. Ensure they are identical.")
        return

    mapping = {np.nan: 0, 'dne': 0, 'positive': 1, 'neutral': 2, 'negative': 3}
    y_arrays = []

    try:
        for df in dataframes:
            y = df[all_keys].replace(mapping).fillna(0).astype(np.uint8).values
            y_arrays.append(y)
    except KeyError as e:
        print(f"Missing expected aspect column: {e}")
        return
    except Exception as e:
        print(f"Error converting labels: {e}")
        return

    y1, y2, y3 = y_arrays[0], y_arrays[1], y_arrays[2]
    y4 = y_arrays[3] if file4 is not None else None

    # 5. Tính F1 score (micro)
    interagree1 = custom_f1_score(y1, y2)
    interagree2 = custom_f1_score(y2, y3)
    interagree3 = custom_f1_score(y3, y1)
    interagree = (interagree1 + interagree2 + interagree3) / 3

    print(f"=== IAA between Annotators ===")
    print(f"Inter-Annotator Agreement 1↔2: {interagree1:.4f}")
    print(f"Inter-Annotator Agreement 2↔3: {interagree2:.4f}")
    print(f"Inter-Annotator Agreement 3↔1: {interagree3:.4f}")
    print(f"Inter-Annotator Agreement Average: {interagree:.4f}")

    if file4 is not None:
        benchmark1 = custom_f1_score(y4, y1)
        benchmark2 = custom_f1_score(y4, y2)
        benchmark3 = custom_f1_score(y4, y3)
        benchmark = (benchmark1 + benchmark2 + benchmark3) / 3

        print()
        print("=== IAA between Annotator vs Gold ===")
        print(f"Annotator 1 F1 micro: {benchmark1:.4f}")
        print(f"Annotator 2 F1 micro: {benchmark2:.4f}")
        print(f"Annotator 3 F1 micro : {benchmark3:.4f}")
        print(f"Annotator Average F1 micro: {benchmark:.4f}")

In [ ]:
def upload_file(nums=3, text=False):
    # ---- Annotator 1 ----
    print("📥 Bước 1: Chọn file Annotator 1 Data")
    uploaded1 = files.upload()
    file1_name = list(uploaded1.keys())[0]
    file1 = json.loads(uploaded1[file1_name].decode('utf-8')) if text==False else uploaded1[file1_name].decode('utf-8')
    print(f"✅ Annotator 1 Data đã tải: {file1_name}\n")

    # ---- Annotator 2 ----
    print("📥 Bước 2: Chọn file Annotator 2 Data")
    uploaded2 = files.upload()
    file2_name = list(uploaded2.keys())[0]
    file2 = json.loads(uploaded2[file2_name].decode('utf-8')) if text==False else uploaded2[file2_name].decode('utf-8')
    print(f"✅ Annotator 2 Data đã tải: {file2_name}\n")

    # ---- Annotator 3 ----
    print("📥 Bước 3: Chọn file Annotator 3 Data (.json)")
    uploaded3 = files.upload()
    file3_name = list(uploaded3.keys())[0]
    file3 = json.loads(uploaded3[file3_name].decode('utf-8')) if text==False else uploaded3[file3_name].decode('utf-8')
    print(f"✅ Annotator 3 Data đã tải: {file3_name}\n")

    if nums == 3:
        return file1, file2, file3

    # ---- Ground Truth ----
    print("📥 Bước 4: Chọn file Goal Data")
    uploaded4 = files.upload()
    file4_name = list(uploaded4.keys())[0]
    file4 = uploaded4[file4_name].decode('utf-8')
    print(f"✅ Goal Data đã tải: {file4_name}\n")

    return file1, file2, file3, file4

# **Round 1**

In [ ]:
file1, file2, file3, file4 = upload_file(nums=4)

📥 Bước 1: Chọn file Annotator 1 Data (.json)


Saving annotations_Dung.json to annotations_Dung.json
✅ Annotator 1 Data đã tải: annotations_Dung.json

📥 Bước 2: Chọn file Annotator 2 Data (.json)


Saving annotations_Thuc.json to annotations_Thuc.json
✅ Annotator 2 Data đã tải: annotations_Thuc.json

📥 Bước 3: Chọn file Annotator 3 Data (.json)


Saving annotations_Tien.json to annotations_Tien.json
✅ Annotator 3 Data đã tải: annotations_Tien.json

📥 Bước 4: Chọn file Goal Data (Ground Truth)


Saving round1_Gold.txt to round1_Gold.txt
✅ Goal Data đã tải: round1_Gold.txt



In [ ]:
pd.set_option('future.no_silent_downcasting', True)

calc_IAA(file1, file2, file3, file4, all_keys)

=== IAA between Annotators ===
Inter-Annotator Agreement 1↔2: 0.6441
Inter-Annotator Agreement 2↔3: 0.8571
Inter-Annotator Agreement 3↔1: 0.6667
Inter-Annotator Agreement Average: 0.7226

=== IAA between Annotator vs Gold ===
Annotator 1 F1 micro: 0.6571
Annotator 2 F1 micro: 0.6301
Annotator 3 F1 micro : 0.7297
Annotator Average F1 micro: 0.6723


# **Round 2**

In [ ]:
file1_02, file2_02, file3_02, file4_02 = upload_file(nums=4)

📥 Bước 1: Chọn file Annotator 1 Data (.json)


Saving annotations_Dung_02.json to annotations_Dung_02.json
✅ Annotator 1 Data đã tải: annotations_Dung_02.json

📥 Bước 2: Chọn file Annotator 2 Data (.json)


Saving annotations_Thuc_02.json to annotations_Thuc_02.json
✅ Annotator 2 Data đã tải: annotations_Thuc_02.json

📥 Bước 3: Chọn file Annotator 3 Data (.json)


Saving annotations_Tien_02.json to annotations_Tien_02.json
✅ Annotator 3 Data đã tải: annotations_Tien_02.json

📥 Bước 4: Chọn file Goal Data (Ground Truth)


Saving round2_Gold.txt to round2_Gold.txt
✅ Goal Data đã tải: round2_Gold.txt



In [ ]:
pd.set_option('future.no_silent_downcasting', True)

calc_IAA(file1_02, file2_02, file3_02, file4_02, all_keys)

=== IAA between Annotators ===
Inter-Annotator Agreement 1↔2: 0.8864
Inter-Annotator Agreement 2↔3: 0.9438
Inter-Annotator Agreement 3↔1: 0.8989
Inter-Annotator Agreement Average: 0.9097

=== IAA between Annotator vs Gold ===
Annotator 1 F1 micro: 0.7723
Annotator 2 F1 micro: 0.8317
Annotator 3 F1 micro : 0.8039
Annotator Average F1 micro: 0.8026


# **Round 3**

In [ ]:
file1_03, file2_03, file3_03, file4_03 = upload_file(nums=4)

📥 Bước 1: Chọn file Annotator 1 Data (.json)


Saving annotations_Dung_03.json to annotations_Dung_03.json
✅ Annotator 1 Data đã tải: annotations_Dung_03.json

📥 Bước 2: Chọn file Annotator 2 Data (.json)


Saving annotations_Thuc_03.json to annotations_Thuc_03.json
✅ Annotator 2 Data đã tải: annotations_Thuc_03.json

📥 Bước 3: Chọn file Annotator 3 Data (.json)


Saving annotations_Tien_03.json to annotations_Tien_03.json
✅ Annotator 3 Data đã tải: annotations_Tien_03.json

📥 Bước 4: Chọn file Goal Data (Ground Truth)


Saving round3_Gold.txt to round3_Gold.txt
✅ Goal Data đã tải: round3_Gold.txt



In [ ]:
pd.set_option('future.no_silent_downcasting', True)

calc_IAA(file1_03, file2_03, file3_03, file4_03, all_keys)

=== IAA between Annotators ===
Inter-Annotator Agreement 1↔2: 0.9714
Inter-Annotator Agreement 2↔3: 0.9515
Inter-Annotator Agreement 3↔1: 0.9423
Inter-Annotator Agreement Average: 0.9551

=== IAA between Annotator vs Gold ===
Annotator 1 F1 micro: 0.9286
Annotator 2 F1 micro: 0.9369
Annotator 3 F1 micro : 0.8909
Annotator Average F1 micro: 0.9188
